In [1]:
# pip install pinecone

NameError: name 'pip' is not defined

In [2]:
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()
# 生产环境中切勿使用硬编码
api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)



## Pinecone的查询配置信息
* 创建索引
    * 确定使用模型
    * 云服务商
    * 相似度度量方法
* 命名空间信息
* 索引信息

In [3]:
index_name = "developer-quickstart-py"

# 配置文件 使用的嵌入模型 云服务 查询方式(默认为余弦)等
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        # 相似度度量方法：cosine, euclidean, dotproduct
        metric="cosine",
        cloud="aws",
        region="us-east-1",
        embed={
            "model": "llama-text-embed-v2",
            "field_map": {"text": "chunk_text"}
        }
)

In [4]:

# 命名空间(namespace)为:developer-quickstart-py
index = pc.Index(index_name)

# 导入数据
records = [
    {"_id": "rec1", "chunk_text": "The Eiffel Tower was completed in 1889 and stands in Paris, France.",
     "category": "history"},
    {"_id": "rec2", "chunk_text": "Photosynthesis allows plants to convert sunlight into energy.",
     "category": "science"},
    {"_id": "rec3", "chunk_text": "Albert Einstein developed the theory of relativity.", "category": "science"},
    {"_id": "rec4", "chunk_text": "The mitochondrion is often called the powerhouse of the cell.",
     "category": "biology"},
    {"_id": "rec5", "chunk_text": "Shakespeare wrote many famous plays, including Hamlet and Macbeth.",
     "category": "literature"},
    {"_id": "rec6", "chunk_text": "Water boils at 100°C under standard atmospheric pressure.", "category": "physics"},
    {"_id": "rec7", "chunk_text": "The Great Wall of China was built to protect against invasions.",
     "category": "history"},
    {"_id": "rec8", "chunk_text": "Honey never spoils due to its low moisture content and acidity.",
     "category": "food science"},
    {"_id": "rec9", "chunk_text": "The speed of light in a vacuum is approximately 299,792 km/s.",
     "category": "physics"},
    {"_id": "rec10", "chunk_text": "Newton's laws describe the motion of objects.", "category": "physics"}
]

# 插入数据 在ns1的命名空间中插入数据
index.upsert_records("ns1", records)

C:\Users\DELL\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


UpsertResponse(upserted_count=10, _response_info={'raw_headers': {'date': 'Wed, 15 Apr 2026 03:26:40 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '3', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '720', 'x-pinecone-response-duration-ms': '721', 'server': 'envoy'}})

In [5]:
print(index.describe_index_stats())
print(type(index.describe_index_stats()))

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '176',
                                    'content-type': 'application/json',
                                    'date': 'Wed, 15 Apr 2026 03:26:46 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '38',
                                    'x-pinecone-request-latency-ms': '37',
                                    'x-pinecone-response-duration-ms': '39'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'ns1': {'vector_count': 10}},
 'storageFullness': 0.0,
 'total_vector_count': 10,
 'vector_type': 'dense'}
<class 'pinecone.core.openapi.db_data.model.index_description.IndexDescription'>


In [ ]:

# 语义查找 问题:
query = "Famous historical structures and monuments"

# 查找结果:
# 使用余弦相似度查找  详细返回结果构成Response results
results = index.search(
    namespace="ns1",
    query={
        "top_k": 5,
        "inputs": {
            'text': query
        },
    },
    # 仅在查询向量是支持
    # 包含向量值
    # include_value=False,
    # 包含元数据
    # include_metadata=True
)

# 返回json结果 包含很多其他配置信息
print(results)

In [ ]:
# 使用重排名提高精度
reranked_results = index.search(
    namespace="ns1",
    query={
        "top_k": 5,
        "inputs": {
            'text': query
        }
    },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 5,
        "rank_fields": ["chunk_text"]
    },
    # 查询范围,指定查询内容
    fields=["category", "chunk_text"]
)

print(reranked_results)

In [6]:
# 查找对应的向量数据
results_vector = index.query(
    namespace="ns1",
    top_k=5,
    vector=[0.0236663818359375, -0.032989501953125, ..., -0.01041412353515625, 0.0086669921875],
    include_values=True,
    include_metadata=True,
)
print(results_vector)

AttributeError: 'ellipsis' object has no attribute '_data_store'

In [7]:
# 根据id内容查询
results_id = index.query(
    namespace="ns1",
    top_k=5,
    id = "rec1",
    include_values=True,
    include_metadata=True,
)

print(results_id)

QueryResponse(matches=[{'id': 'rec1',
 'metadata': {'category': 'history',
              'chunk_text': 'The Eiffel Tower was completed in 1889 and stands '
                            'in Paris, France.'},
 'score': 1.00006437,
 'values': [0.0467529297,
            -0.0228271484,
            -0.0230102539,
            -0.051574707,
            -0.00354385376,
            0.00969696,
            0.00608825684,
            0.00244903564,
            -0.0681152344,
            0.033203125,
            0.0115280151,
            -0.00942993164,
            -0.0365600586,
            -0.0243377686,
            -0.0370178223,
            0.0285797119,
            0.0107192993,
            -0.00907135,
            0.0182800293,
            -0.0382995605,
            -0.041229248,
            0.0386047363,
            0.00587081909,
            -0.017288208,
            -0.0569458,
            -0.0811157227,
            0.018081665,
            -0.0189361572,
            0.0240936279,
         